# Лабораторная работа №6
## Параметризованное сканирование β в SIR-модели на сети Петри

В данном скрипте выполняется расширенное параметрическое исследование
SIR-модели, представленной в виде сети Петри.

Базовый скрипт `sirpetri_scan_parameters.jl` исследовал зависимость
показателей `peak_I` и `final_R` от коэффициента заражения `β` при одном
фиксированном значении коэффициента выздоровления `γ = 0.1`.

В этой версии сохраняется та же идея сканирования `β`, но дополнительно
рассматриваются три значения `γ`.

Это позволяет оценить, как скорость выздоровления влияет на:

- максимальное число инфицированных;
- итоговое число выздоровевших;
- выраженность вспышки при разных значениях коэффициента заражения.

Чтобы не перегружать отчёт, формируется только три итоговых рисунка —
по одному для каждого значения `γ`.

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение модели

Основная реализация модели SIR в подходе сетей Петри подключается из файла
`src/SIRPetri.jl`.

In [ ]:
include(srcdir("SIRPetri.jl"))
using .SIRPetri

using DataFrames, CSV, Plots

## Диапазон коэффициента заражения β

Как и в базовом скрипте, коэффициент заражения `β` изменяется в диапазоне
от `0.1` до `0.8` с шагом `0.05`.

In [ ]:
β_range = 0.1:0.05:0.8
tmax = 100.0

## Наборы параметров γ

Для сравнения выбраны три значения коэффициента выздоровления:

- `γ = 0.05` — медленное выздоровление;
- `γ = 0.1` — базовый вариант из методички;
- `γ = 0.2` — более быстрое выздоровление.

Такой выбор позволяет проверить, как изменение скорости выздоровления
влияет на результаты сканирования по `β`.

In [ ]:
gamma_sets = [
    (
        name = "gamma_low",
        γ = 0.05,
        description = "Медленное выздоровление",
    ),
    (
        name = "gamma_base",
        γ = 0.1,
        description = "Базовое выздоровление",
    ),
    (
        name = "gamma_high",
        γ = 0.2,
        description = "Быстрое выздоровление",
    ),
]

## Общая сводная таблица

В эту таблицу будут добавлены результаты всех прогонов.
Она нужна для последующего анализа и для проверки численных значений.

In [ ]:
summary = DataFrame(
    scenario = String[],
    gamma = Float64[],
    β = Float64[],
    peak_I = Float64[],
    final_R = Float64[],
)

println("Запуск параметризованного сканирования β...")

## Основной цикл параметрического исследования

Для каждого значения `γ` выполняется отдельное сканирование диапазона `β`.
Для каждой пары `(β, γ)` запускается детерминированная симуляция SIR-модели.

После каждого запуска вычисляются:

- `peak_I` — максимальное число инфицированных;
- `final_R` — конечное число выздоровевших.

In [ ]:
for scenario in gamma_sets
    println("="^60)
    println("Сценарий: ", scenario.description)
    println("γ = ", scenario.γ)

    results = []

    for β in β_range
        net, u0, _ = build_sir_network(β, scenario.γ)

        df = simulate_deterministic(
            net,
            u0,
            (0.0, tmax),
            saveat = 0.5,
            rates = [β, scenario.γ],
        )

        peak_I = maximum(df.I)
        final_R = df.R[end]

        push!(
            results,
            (
                β = Float64(β),
                peak_I = Float64(peak_I),
                final_R = Float64(final_R),
            ),
        )

        push!(
            summary,
            (
                scenario.name,
                Float64(scenario.γ),
                Float64(β),
                Float64(peak_I),
                Float64(final_R),
            ),
        )
    end

## Таблица для одного сценария

Для каждого значения `γ` сохраняется отдельная таблица.
Все файлы сохраняются прямо в каталог `data/`.

In [ ]:
    df_scan = DataFrame(results)

    csv_file = datadir("sir_scan_$(scenario.name).csv")
    CSV.write(csv_file, df_scan)

## Построение графика

Для каждого значения `γ` строится один график зависимости
`peak_I` и `final_R` от `β`.

Таким образом, всего сохраняется три рисунка.

In [ ]:
    p = plot(
        df_scan.β,
        [df_scan.peak_I df_scan.final_R],
        label = ["Peak I" "Final R"],
        marker = :circle,
        xlabel = "β (infection rate)",
        ylabel = "Population",
        title = "$(scenario.description), γ = $(scenario.γ)",
        linewidth = 2,
    )

    fig_file = plotsdir("sir_scan_$(scenario.name).png")
    savefig(p, fig_file)

    println("Сохранено:")
    println("  ", csv_file)
    println("  ", fig_file)
end

## Сохранение общей сводной таблицы

В общей таблице объединяются результаты всех трёх сценариев.
Это удобно для итогового анализа и сравнения.

In [ ]:
summary_file = datadir("sir_scan_params_summary.csv")
CSV.write(summary_file, summary)

println("="^60)
println("Параметризованное сканирование завершено.")
println("Сводная таблица сохранена в: ", summary_file)
println("Рисунки сохранены в каталог: ", plotsdir())

## Итог

После выполнения скрипта создаются:

- `data/sir_scan_gamma_low.csv`;
- `data/sir_scan_gamma_base.csv`;
- `data/sir_scan_gamma_high.csv`;
- `data/sir_scan_params_summary.csv`;
- `plots/sir_scan_gamma_low.png`;
- `plots/sir_scan_gamma_base.png`;
- `plots/sir_scan_gamma_high.png`.

Такая версия позволяет сравнить влияние коэффициента выздоровления `γ`
на зависимость пика инфицированных и конечного числа выздоровевших
от коэффициента заражения `β`.